# Measure It — U-Net training on Colab GPU

**Repo:** [kashifmayar/measure-it](https://github.com/kashifmayar/measure-it) branch `dev-test01`

This notebook trains the pen **U-Net** and runs **test-set evaluation**. It does **not** upload your photos to GitHub.

**You must provide locally (not in git):**
- `dataset/images/*.jpg` (undistorted)
- `dataset/masks/*.png`
- `dataset/splits/*.txt` (already in repo if you clone full branch)

**Runtime:** Menu → **Runtime → Change runtime type → GPU (T4)**

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Clone repository

In [ ]:
!rm -rf measure-it
!git clone -b dev-test01 --depth 1 https://github.com/kashifmayar/measure-it.git
%cd measure-it

## 2. Install dependencies

Colab includes PyTorch; we install OpenCV contrib + HEIC support.

In [ ]:
!pip install -q opencv-contrib-python pillow-heif matplotlib

## 3. Dataset — **do not use Colab browser upload** (too slow)

**Best:** Upload zip from PC to **Google Drive** (Drive desktop app syncs in background). Then set `USE_DRIVE = True` and path below.

**Smaller zip on PC:** `scripts\pack_dataset_for_colab_small.ps1` → upload `dataset_colab_upload_small.zip` to Drive.

**Already uploading?** Let Drive finish, then run the Drive cell only — skip the Colab `files.upload()` cell.

In [ ]:
from pathlib import Path
import zipfile

ROOT = Path('/content/measure-it')
DATASET = ROOT / 'dataset'
(DATASET / 'images').mkdir(parents=True, exist_ok=True)
(DATASET / 'masks').mkdir(parents=True, exist_ok=True)

# --- Option A: Drive ---
USE_DRIVE = False  # set True after mounting
DRIVE_ZIP_PATH = '/content/drive/MyDrive/measure-it/dataset_images_masks.zip'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    assert Path(DRIVE_ZIP_PATH).is_file(), f'Missing {DRIVE_ZIP_PATH}'
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall(ROOT)  # zip must contain dataset/images and dataset/masks
    print('Extracted from Drive')
else:
    print('Skip Drive — use upload cell below or set USE_DRIVE=True')

In [ ]:
# --- Option B: upload zip to Colab ---
from google.colab import files
import zipfile
from pathlib import Path

ROOT = Path('/content/measure-it')
DATASET = ROOT / 'dataset'

uploaded = files.upload()  # select dataset_images_masks.zip
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name, 'r') as z:
            z.extractall(ROOT)
        print('Extracted', name)

n_img = len(list((DATASET / 'images').glob('*.jpg')))
n_msk = len(list((DATASET / 'masks').glob('*.png')))
print('images:', n_img, 'masks:', n_msk)
assert n_img >= 70 and n_msk >= 70, 'Need images+masks in dataset/ after unzip'

In [ ]:
!python scripts/fix_dataset_layout.py
!python scripts/prepare_dataset.py validate

## 4. Train U-Net (GPU)

Adjust epochs/batch if you hit OOM (reduce `batch-size` or `image-size`).

In [ ]:
!python models/train.py \
  --device cuda \
  --epochs 30 \
  --batch-size 8 \
  --image-size 384 \
  --lr 1e-3 \
  --seed 42

In [ ]:
from IPython.display import Image, display
display(Image(filename='models/runs/unet_pen/training_curves.png'))

## 5. Evaluate on **test** split only

In [ ]:
!python models/evaluate.py \
  --checkpoint models/runs/unet_pen/best.pt \
  --split test \
  --out validation/test_metrics_colab.json

!cat validation/test_metrics_colab.json

## 6. Download `best.pt` to your PC

Copy to `E:\Project\measure it\models\runs\unet_pen\best.pt` for `main.py --checkpoint`.

Update README SHA-256: run `scripts/hash_checkpoint.ps1` on Windows after download.

In [ ]:
import hashlib
from pathlib import Path
from google.colab import files

ckpt = Path('models/runs/unet_pen/best.pt')
assert ckpt.is_file(), 'Train first'
h = hashlib.sha256(ckpt.read_bytes()).hexdigest()
print('SHA-256:', h)
files.download(str(ckpt))

# Optional: save to Drive
# !cp models/runs/unet_pen/best.pt /content/drive/MyDrive/measure-it/best.pt

## 7. (Optional) Zip full run for backup

Includes `best.pt`, curves, config, test metrics.

In [ ]:
!zip -r colab_unet_run.zip models/runs/unet_pen validation/test_metrics_colab.json
from google.colab import files
files.download('colab_unet_run.zip')